In [ ]:
!pip install transformers datasets torch scikit-learn

# Данные, аугментация

Готовый датасет: https://drive.google.com/file/d/1eheqUrARZ4exdHYREab6WMIj1ZhSsaV8/view?usp=sharing

In [ ]:
import torch
from transformers import pipeline

# Указываем устройство: "cuda" для GPU, "cpu" для CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Загружаем модель для перефразирования
paraphraser = pipeline("text2text-generation", model="cointegrated/rut5-base-paraphraser", device=0 if torch.cuda.is_available() else -1)

# Функция перефразирования
def paraphrase_sentences(sentences, max_length=50):
    result = paraphraser(sentences, max_length=max_length)
    return [res["generated_text"] for res in result]

/usr/local/lib/python3.10/dist-packages/transformers/convert_slow_tokenizer.py:561: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [3]:
import json
import pandas as pd
import random
random.seed(42)

In [ ]:
with open("helpers.json", "r", encoding="utf-8") as f:
    helpers = json.load(f)

with open("neutral_helpers.json", "r", encoding="utf-8") as f:
    neutral_helpers = json.load(f)

csvs = [("https://docs.google.com/spreadsheets/d/e/2PACX-1vT1OUTDKTAab-GnpO1fMJTbhKeAE0PDxUOaphiyMHlR93S43N-rMHoPT_1IDVPYtVR0EO7s5y2-ChME/pub?gid=1580057433&single=true&output=csv", "Преимущества", 100),
        ("https://docs.google.com/spreadsheets/d/e/2PACX-1vSWa5tNnSmBYf-_Xef5BPWkIzsRrhHjmdXLLqxLuE40tNOSSzPTRMb3y8qcfvukKHeSvv_KvsDu_H-1/pub?gid=1120824874&single=true&output=csv", "Недостатки", 100),
        ("https://docs.google.com/spreadsheets/d/e/2PACX-1vRyQwkZ-xSxwQY3u-sg8nbA9o-HqpKE0cA8BTJH6fYUTxPFyqeyqbQkPuURFHm32KuyEDP9r771Duxb/pub?gid=1980537576&single=true&output=csv", "", 256)]

dfs = []
augmented_dfs = []
to_mark = []

with_mark_query = "Скорость != 0 | Цена != 0 | Камера != 0"
for csv, support_class, max_token in csvs:
    df = pd.read_csv(csv)
    col = df.columns[0]
    df = df[~df[col].isna()]
    df = df.fillna(0).reset_index(drop=True)
    index = df.query(with_mark_query).last_valid_index()

    no_marks_df = df[df.index > index]
    marks_df = df[df.index <= index]

    if support_class:
        augmented_df = marks_df.query(with_mark_query)
        augmented_df[col] = paraphrase_sentences(augmented_df[col].tolist(), max_token)
        augmented_dfs.append(augmented_df.rename(columns={col: "comment"}))

        marks_df[col] = marks_df[col].apply(lambda x: random.choice(helpers.get(support_class)) + " : " + x)
        no_marks_df[col] = no_marks_df[col].apply(lambda x: random.choice(neutral_helpers.get(support_class)) + " : " + x)

    dfs.append(marks_df.rename(columns={col: "comment"}))
    to_mark.append(no_marks_df.rename(columns={col: "comment"}))

In [ ]:
with open("to_mark.json", "w", encoding="utf-8") as f:
    json.dump(pd.concat(to_mark).to_dict("records"), f, ensure_ascii=False, indent=4)

In [ ]:
# Полностью искусственные комментарии
genders = {
    "m": "ый", "f": "ая", "it": "ое", "mult": "ие"
}

data = {
    "Скорость": {
        "positive": ["быстр", "мощн", "надежн", "энергоэффективн", "современн", "оптимизированн", "производительн", "плавн", "стабильн", "впечатляющ"],
        "negative": ["медленн", "слаб", "ненадежн", "устаревш", "нестабильн", "неэффективн", "разочаровывающ"],
        "nouns": {
            "m": ["чип", "процессор"],
            "f": ["производительность"],
            "it": ["ядро"],
        }
    },
    "Камера": {
        "positive": ["четк", "качественн", "детализированн", "ярк", "светосильн", "профессиональн", "продвинут", "резк", "великолепн", "насыщенн"],
        "negative": ["размыт", "тускл", "слаб", "посредственн", "устаревш", "нечетк", "дефектн", "низкокачественн"],
        "nouns": {
            "f": ["камера", "фотография", "цветопередача", "детализация", "съемка"],
            "it": ["изображение", "качество"],
            "mult": ["фотки", "снимки", "изображения", "фоточки", "фотографии"]
        }
    },
    "Цена": {
        "positive": ["доступн", "справедлив", "выгодн", "оптимальн", "приемлем", "разумн", "честн", "адекватн", "привлекательн", "сбалансированн"],
        "negative": ["завышенн", "высок", "необоснованн", "абсурдн", "необоснованно высок", "слишком высок"],
        "nouns": {
            "m": ["ценник"],
            "f": ["цена", "стоимость", "трата"]
        }
    }
}

connectors = {
    "positive": ["приятно удивляет", "радует", "вызывает восхищение", "достойна похвалы",
                 "оправдывает ожидания", "соответствует заявленным характеристикам", "подчеркивает преимущества"],
    "negative": ["оставляет желать лучшего", "вызывает разочарование", "не оправдывает ожиданий",
                 "значительно уступает конкурентам", "портит общее впечатление", "мешает комфорту использования",
                 "создает проблемы"]
}

beginners = [
    "Ща все расскажу", "Вот мой личный опыт использования", "Итак, вот что я могу сказать после месяца использования",
    "Хотел бы с вами поделиться своими впечатлениями от использования данного девайса",
    "Хочу поделиться своими мыслями о данном устройстве",
    "Почитал я отзывы и решил написать свой", "Думаете брать? Знайте", "Обязательно к прочтению перед покупкой",
    "Покупка была совершена где-то пару недель назад", "Я думаю вот что", "Я считаю",
    "Обзавелся данным устройством и вот что могу сказать про него",
    "Мое личное мнение об этом девайсе",
]
enders = [
    "Ну наверное как-то так.", "В целом все, что могу сказать.", "Ну в общем вы поняли.",
    "Вот что я думаю об этом товаре.", "На этом у меня всё", "Думайте сами, брать или не брать.",
    "Задумайтесь!", "Знал бы я об этом раньше!", "Короче говоря вы поняли",
    "Надеюсь коммент будет для вас полезен", "Много кэшбэка пришло с покупки)))",
    "Нашел дешевле на другом сайте уже после покупки... Облоооом",
    "Через неделю после покупки цены подлетели - повезло?",
]

cl = ["positive", "negative"]

variants = {}
for aspect in data:
    combs = []
    for gender, nouns in data[aspect]["nouns"].items():
        for noun in nouns:
            for label in cl:
                mark = 1 if label == "positive" else -1
                for adj_ in data[aspect][label]:
                    adj = adj_ + genders[gender]
                    if random.choice([True, False]):
                        combs.append((f"{adj} {noun} {random.choice(connectors[label])}", mark))
                    else:
                        combs.append((f"{noun} {adj}", mark))
    variants[aspect] = combs

combs = []
for val in variants["Камера"]:
    for j in range(2):
        labels = []
        chosen_cam, lab_cam = val
        chosen_proc, lab_proc = random.choice(variants["Скорость"])
        chosen_price, lab_price = random.choice(variants["Цена"])
        comb = [chosen_proc, chosen_cam, chosen_price]

        will_be_taken = [0, 1, 2]
        random.shuffle(will_be_taken)
        will_be_taken = will_be_taken[:random.choice([2, 3])]

        comb = [comb[i] for i in will_be_taken]

        if 0 not in will_be_taken: lab_proc = 0
        if 1 not in will_be_taken: lab_cam = 0
        if 2 not in will_be_taken: lab_price = 0

        text = ", ".join(comb)
        flag = False
        if random.choice([True, True, True, True, False, False, False]):
            text += ". " + random.choice(enders)
            flag = True

        if random.choice([True, True, True, True, False, False, False]):
            text = random.choice(beginners) + " : " + text
            flag = True

        if flag or len(will_be_taken) == 3:
            combs.append((text, lab_proc, lab_cam, lab_price))

In [ ]:
augmented_dfs.append(pd.DataFrame(combs, columns=augmented_dfs[-1].columns).drop_duplicates(subset=["comment"]))

In [ ]:
augmented_dfs

[                                               comment  Скорость  Камера  Цена
 0    Загружается быстро, все нужное и не нужное, ра...       1.0     0.0   0.0
 1                  Телефон полностью оправдывает цену.       0.0     0.0   1.0
 3    1. Очень шустро работает для бюджета, пока не ...       1.0     0.0   1.0
 4    Классный смартфон. Мне нравится.Это мой первый...       1.0     0.0   0.0
 5    Цена дешевая. Качество даже выше для 6500 руб....       0.0     0.0   1.0
 ..                                                 ...       ...     ...   ...
 449                    Отличный телефон за свои деньги       0.0     0.0   1.0
 451                                Хорошо за свою цену       0.0     0.0   1.0
 452  Быстрый, батарея хорошая, собеседник хорошо ме...       1.0     0.0   0.0
 453            Хороший экран с большим запасом яркости       1.0     0.0   0.0
 456  Хороший вид, долго искал где розовый вариант в...       1.0     0.0   0.0
 
 [326 rows x 4 columns],
             

In [ ]:
data = pd.concat(dfs + augmented_dfs)
with open("dataset.json", "w", encoding="utf-8") as f:
    json.dump(data.to_dict("records"), f, ensure_ascii=False, indent=4)

In [ ]:
for col in ["Скорость", "Камера", "Цена"]:
    print(col, len(data[data[col] == 0]))

Скорость 1013
Камера 1135
Цена 1089


In [ ]:
for col in ["Скорость", "Камера", "Цена"]:
    print(col, len(data[data[col] != 0]))

Скорость 862
Камера 740
Цена 786


# Обучение

In [4]:
with open("dataset.json", "r", encoding="utf-8") as f:
    data = json.load(f)

data = pd.DataFrame(data)

In [5]:
from sklearn.model_selection import train_test_split


reviews = data["comment"].values
labels = data[["Скорость", "Камера", "Цена"]].values

X_train, X_test, y_train, y_test = train_test_split(reviews, labels, test_size=0.2, random_state=42)

In [6]:
from transformers import AutoTokenizer


# Загрузка модели и токенизатора
model_name = "DeepPavlov/rubert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)


train_encodings = tokenizer(X_train.tolist(), truncation=True, padding=True, max_length=128, return_tensors="pt")
test_encodings = tokenizer(X_test.tolist(), truncation=True, padding=True, max_length=128, return_tensors="pt")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.65M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [7]:
import torch
from torch.utils.data import DataLoader


class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        # Вытаскиваем токенизированные данные для конкретного индекса
        item = {key: val[idx] for key, val in self.encodings.items()}
        # Добавляем метки (float для регрессии)
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float32)
        return item


train_loader = DataLoader(CustomDataset(train_encodings, y_train), batch_size=16)
val_loader = DataLoader(CustomDataset(test_encodings, y_test))

In [8]:
import numpy as np
from sklearn.metrics import f1_score
import torch
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer, AdamW, get_scheduler

# Убедимся, что у нас доступен GPU (если есть)
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")


# Загружаем RuBERT и токенизатор
model_name = "DeepPavlov/rubert-base-cased"
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)  # Выходной слой на 3 класса
tokenizer = AutoTokenizer.from_pretrained(model_name)

model.to(device)  # Переносим модель на GPU (если доступен)

# Оптимизатор и scheduler
optimizer = AdamW(model.parameters(), lr=7e-5)
num_epochs = 5
num_training_steps = num_epochs * len(train_loader)
lr_scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)

# Функция потерь (так как у нас метки -1, 0, 1, используем MSELoss для регрессии)
criterion = torch.nn.MSELoss()

# Правило преобразования логитов в классы
def postprocess_logits(logits):
    classes = np.where(logits < -0.33, -1, np.where(logits > 0.33, 1, 0))
    return classes


# Функция обучения
def train_epoch(model, train_loader, optimizer, scheduler, criterion):
    model.train()
    total_loss = 0

    for batch in train_loader:
        # Переносим данные на GPU
        inputs = {key: val.to(device) for key, val in batch.items() if key != "labels"}
        labels = batch["labels"].to(device)

        # Прямой проход
        optimizer.zero_grad()
        outputs = model(**inputs)
        logits = outputs.logits

        # Считаем loss и обновляем веса
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
    return total_loss / len(train_loader)


# Функция валидации
def evaluate(model, val_loader, criterion):
    model.eval()

    total_loss = 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in val_loader:
            inputs = {key: val.to(device) for key, val in batch.items() if key != "labels"}
            labels = batch["labels"].to(device)

            outputs = model(**inputs)
            logits = outputs.logits
            loss = criterion(logits, labels)

            total_loss += loss.item()

            labels = labels.cpu().numpy()
            all_labels.append(labels)
            logits = logits.cpu().numpy()
            preds = postprocess_logits(logits)  # Преобразуем логиты в классы
            all_preds.append(preds)

    # Объединяем все предсказания и метки
    all_preds = np.concatenate(all_preds, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)

    # Рассчитываем F1-меру по каждому классу
    f1_per_class = []
    for i in range(all_labels.shape[1]):  # Для каждого класса
        f1 = f1_score(all_labels[:, i], all_preds[:, i], average="macro")
        f1_per_class.append(f1)

    return total_loss / len(val_loader), f1_per_class

try:
  initial_val_loss = 1e10
  # Цикл обучения
  for epoch in range(num_epochs):
      train_loss = train_epoch(model, train_loader, optimizer, lr_scheduler, criterion)
      val_loss, f1_all_classes = evaluate(model, val_loader, criterion)

      print(f"Epoch {epoch + 1}")
      print(f"Train Loss: {train_loss:.4f}")
      print(f"Validation Loss: {val_loss:.4f}")
      print(f"f1 for all classes: {f1_all_classes}")

      if val_loss < initial_val_loss:
          initial_val_loss = val_loss

          # Сохранение модели
          model.save_pretrained("rubert_multilabel_model-best")
          tokenizer.save_pretrained("rubert_multilabel_model-best")
finally:
  if val_loss != initial_val_loss:
      model.save_pretrained("rubert_multilabel_model-last")
      tokenizer.save_pretrained("rubert_multilabel_model-last")

pytorch_model.bin:   0%|          | 0.00/714M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at DeepPavlov/rubert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoch 1
Train Loss: 0.3579
Validation Loss: 0.2784
f1 for all classes: [0.6547965692340793, 0.5244905238666692, 0.5566911940137093]
Epoch 2
Train Loss: 0.2335
Validation Loss: 0.1793
f1 for all classes: [0.7835369732375946, 0.633743388939077, 0.8567182137924161]
Epoch 3
Train Loss: 0.1236
Validation Loss: 0.1135
f1 for all classes: [0.8361657666005492, 0.8398118221647634, 0.8705404613319621]
Epoch 4
Train Loss: 0.0716
Validation Loss: 0.0952
f1 for all classes: [0.864624058068869, 0.8835721082912094, 0.8775410749094958]
Epoch 5
Train Loss: 0.0488
Validation Loss: 0.0877
f1 for all classes: [0.902701688732778, 0.9144446072326814, 0.8957642343836373]


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Загрузка модели и токенизатора
model = AutoModelForSequenceClassification.from_pretrained("rubert_multilabel_model-best")
tokenizer = AutoTokenizer.from_pretrained("rubert_multilabel_model-best")

In [ ]:
text = "преимущества: снимки ОГОНЬ!!!"
inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
outputs = model(**inputs)
predictions = outputs.logits.detach().cpu().numpy()
predictions

array([[-0.19514492,  0.9736891 , -0.07791621]], dtype=float32)

In [9]:
!zip -r rubert_multilabel_model-best.zip rubert_multilabel_model-best

  adding: rubert_multilabel_model-best/ (stored 0%)
  adding: rubert_multilabel_model-best/config.json (deflated 54%)
  adding: rubert_multilabel_model-best/tokenizer.json (deflated 73%)
  adding: rubert_multilabel_model-best/model.safetensors (deflated 7%)
  adding: rubert_multilabel_model-best/tokenizer_config.json (deflated 75%)
  adding: rubert_multilabel_model-best/special_tokens_map.json (deflated 42%)
  adding: rubert_multilabel_model-best/vocab.txt (deflated 64%)


In [ ]:
from google.colab import files
files.download("rubert_multilabel_model-best.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import numpy as np

def postprocess_logits(logits  ):
    # Преобразуем числа в классы
    classes = np.where(logits < -0.33, -1, np.where(logits > 0.33, 1, 0))
    return classes